# ENARES 2024 CRS04 — Stage 03
## Notebook 01A · Construcción estructural de la tabla analytical

Este notebook solo crea la tabla analytical base desde cleaned, preserva la llave y agrega `ID_AULA`. No construye indicadores.


In [ ]:
!pip install -q google-cloud-bigquery pandas pandas-gbq pyarrow db-dtypes openpyxl XlsxWriter tabulate
from google.colab import auth, drive
from google.cloud import bigquery
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd, hashlib, os
auth.authenticate_user(); drive.mount('/content/drive')
PROJECT_ID='enares-2024-crs04'; LOCATION='US'; EXPECTED_ROWS=18807
ROOT_DRIVE=Path('/content/drive/MyDrive/ENARES_2024_PROJECT')
LOG_DIR=ROOT_DRIVE/'05Resultados'/'logs'/'stage03'; SQL_DIR=ROOT_DRIVE/'02SQL'; OUTPUT_DIR=ROOT_DRIVE/'04Outputs'; DOCS_DIR=ROOT_DRIVE/'docs'; R_DIR=ROOT_DRIVE/'03Scripts_R'
for d in [LOG_DIR,SQL_DIR,OUTPUT_DIR,DOCS_DIR,R_DIR]: d.mkdir(parents=True,exist_ok=True)
RUN_UTC=datetime.now(timezone.utc).isoformat(); client=bigquery.Client(project=PROJECT_ID,location=LOCATION)
display(client.query('SELECT CURRENT_DATE() AS fecha_actual').result().to_dataframe())

## 1. Verificar cleaned

In [ ]:
t=client.get_table(f'{PROJECT_ID}.enares2024_crs04_cleaned.cleaned_crs04_merged_adolescents')
if t.num_rows!=EXPECTED_ROWS: raise RuntimeError('Cleaned inválida')

## 2. Skip map

In [ ]:
skip_map=pd.DataFrame([
{'block':'VP_HOGAR','gateway_var':'C3P203','open_value':1,'dependent_vars':'C3P201_1..C3P201_11','recode':'SYSMIS->0','keep_null_when':'C3P203 IS NULL','module':'3.2'},
{'block':'VF_HOGAR','gateway_var':'C3P207','open_value':1,'dependent_vars':'C3P205_1..C3P205_7','recode':'SYSMIS->0','keep_null_when':'C3P207 IS NULL','module':'3.2'},
{'block':'VP_ESCUELA','gateway_var':'C3P225','open_value':1,'dependent_vars':'C3P223_1..C3P223_14','recode':'SYSMIS->0','keep_null_when':'C3P225 IS NULL','module':'3.3'},
{'block':'VF_ESCUELA','gateway_var':'C3P229','open_value':1,'dependent_vars':'C3P227_1..C3P227_10','recode':'SYSMIS->0','keep_null_when':'C3P229 IS NULL','module':'3.3'},
{'block':'VS_12M','gateway_var':'CONFIRMAR_PDF','open_value':1,'dependent_vars':'C4P248_1..C4P248_16','recode':'SYSMIS->0','keep_null_when':'gateway IS NULL','module':'3.4'}])
skip_map.to_csv(LOG_DIR/'stage3_skip_map.csv',index=False); display(skip_map)
client.load_table_from_dataframe(skip_map,f'{PROJECT_ID}.enares2024_crs04_outputs.stage3_skip_map',job_config=bigquery.LoadJobConfig(write_disposition='WRITE_TRUNCATE')).result()

## 3. Dominios

In [ ]:
defs={'C3P301_1':[1,2,3],'C3P301_2':[1,2,3],'C3P301_3':[1,2,3],'C3P301_4':[1,2,3],'C3P301_5':[1,2,3],'C3P301_6':[1,2,3],'C3P128':[1,2,3,4],'C4P129':list(range(1,10))}
rvals=pd.DataFrame([{'variable':v,'value':x} for v,xs in defs.items() for x in xs]); sql='SELECT CAST(variable_name AS STRING) variable_name, CAST(value AS STRING) value FROM `'+PROJECT_ID+'.enares2024_crs04_raw.metadata_crs04_value_labels`'; labels=client.query(sql).result().to_dataframe(); valid=set(zip(labels['variable_name'],labels['value'])); rvals['value_exists']=[(v,str(x)) in valid for v,x in zip(rvals['variable'],rvals['value'])]; rvals.to_csv(LOG_DIR/'stage3_value_domain_check.csv',index=False); display(rvals)

## 4. Crear analytical base

La recreación es intencional y elimina cualquier versión previa de analytical. Por eso este notebook debe ejecutarse antes de todos los notebooks de indicadores.


In [ ]:
CLEAN=f'{PROJECT_ID}.enares2024_crs04_cleaned.cleaned_crs04_merged_adolescents'
A=f'{PROJECT_ID}.enares2024_crs04_analytical.analytical_crs04_adolescents'
clean_schema={f.name for f in client.get_table(CLEAN).schema}
required={'ID','COLEGIAL_ID','C3ANIO','TURNO','C3SECC'}
missing=required-clean_schema
if missing: raise RuntimeError(f'Faltan variables estructurales: {sorted(missing)}')
source_select='* EXCEPT(ID_AULA)' if 'ID_AULA' in clean_schema else '*'
sql=f'''CREATE OR REPLACE TABLE `{A}` AS
SELECT {source_select},
  CONCAT(CAST(ID AS STRING),'_',CAST(C3ANIO AS STRING),'_',CAST(TURNO AS STRING),'_',UPPER(TRIM(CAST(C3SECC AS STRING)))) AS ID_AULA
FROM `{CLEAN}`'''
(SQL_DIR/'stage3_08a_create_analytical_base.sql').write_text(sql,encoding='utf-8')
client.query(sql,location=LOCATION).result()
print('Analytical base creada sin indicadores.')
